In [1]:
import random, os, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

PREPROCESSED_DIR = "/kaggle/input/datasets/iftekharuddin27/preprocessed-datasets"
OUTPUT_DIR       = "/kaggle/working/"
MODEL_NAME       = "xlm-roberta-base"
MAX_LEN, BATCH_SIZE, EPOCHS, LR = 128, 64, 5, 2e-5
ALPHA, BETA, GAMMA = 0.3, 0.3, 0.4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = {0: 'Non-hateful', 1: 'Hateful', 2: 'Sarcastic'}
print(f"Device: {device}")

Device: cuda


In [2]:
en_train = pd.read_csv(f"{PREPROCESSED_DIR}/en_train.csv")
en_val   = pd.read_csv(f"{PREPROCESSED_DIR}/en_val.csv")
en_test  = pd.read_csv(f"{PREPROCESSED_DIR}/en_test.csv")
bn_train = pd.read_csv(f"{PREPROCESSED_DIR}/bn_train.csv")
bn_val   = pd.read_csv(f"{PREPROCESSED_DIR}/bn_val.csv")
bn_test  = pd.read_csv(f"{PREPROCESSED_DIR}/bn_test.csv")

for df in [en_train, en_val, en_test, bn_train, bn_val, bn_test]:
    df['text_clean'] = df['text_clean'].fillna('')

def add_aux_labels(df, label_col):
    df['is_hateful']   = (df[label_col] == 1).astype(int)
    df['is_sarcastic'] = (df[label_col] == 2).astype(int)
    return df

en_train = add_aux_labels(en_train, 'class')
en_val   = add_aux_labels(en_val,   'class')
en_test  = add_aux_labels(en_test,  'class')
bn_train = add_aux_labels(bn_train, 'label')
bn_val   = add_aux_labels(bn_val,   'label')
bn_test  = add_aux_labels(bn_test,  'label')
print("Data loaded.")

Data loaded.


In [3]:
class AblationModel(nn.Module):
    def __init__(self, model_name, num_classes=3,
                 use_hate_head=True, use_sarc_head=True, dropout=0.3):
        super().__init__()
        self.use_hate_head = use_hate_head
        self.use_sarc_head = use_sarc_head
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        if use_hate_head:
            self.hate_head = nn.Sequential(
                nn.Linear(hidden_size, 256), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(256, 1))
        if use_sarc_head:
            self.sarcasm_head = nn.Sequential(
                nn.Linear(hidden_size, 256), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(256, 1))
        fusion_in = hidden_size
        if use_hate_head: fusion_in += 256
        if use_sarc_head: fusion_in += 256
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fusion_in, 512), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
            outputs    = self.encoder(input_ids=input_ids,
                                      attention_mask=attention_mask)
            cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])
            parts      = [cls_output]
            
            # We will store active outputs in a dictionary
            out_dict = {}
    
            if self.use_hate_head:
                hate_hidden = F.relu(self.hate_head[0](cls_output))
                hate_hidden = self.dropout(hate_hidden)
                hate_logit  = self.hate_head[3](hate_hidden).squeeze(-1)
                parts.append(hate_hidden)
                out_dict['hate_logit'] = hate_logit
                
            if self.use_sarc_head:
                sarc_hidden = F.relu(self.sarcasm_head[0](cls_output))
                sarc_hidden = self.dropout(sarc_hidden)
                sarc_logit  = self.sarcasm_head[3](sarc_hidden).squeeze(-1)
                parts.append(sarc_hidden)
                out_dict['sarc_logit'] = sarc_logit
                
            fused         = torch.cat(parts, dim=1)
            logits_3class = self.fusion_mlp(fused)
            
            out_dict['logits_3class'] = logits_3class
            return out_dict

In [4]:
class DualHeadDataset(Dataset):
    def __init__(self, texts, labels_3class, labels_hate,
                 labels_sarc, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_len, return_tensors='pt')
        self.labels_3class = torch.tensor(labels_3class, dtype=torch.long)
        self.labels_hate   = torch.tensor(labels_hate,   dtype=torch.float)
        self.labels_sarc   = torch.tensor(labels_sarc,   dtype=torch.float)

    def __len__(self): return len(self.labels_3class)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels_3class'] = self.labels_3class[idx]
        item['labels_hate']   = self.labels_hate[idx]
        item['labels_sarc']   = self.labels_sarc[idx]
        return item

In [5]:
# ============================================================
#  Ablation training function
# ============================================================
def run_ablation(train_df, val_df, test_df, label_col, lang,
                 use_hate, use_sarc, variant_name):

    print(f"\n{'='*65}")
    print(f"Ablation: {variant_name} — {lang}")
    print(f"  use_hate_head={use_hate}  use_sarc_head={use_sarc}")
    print(f"{'='*65}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def make_ds(df):
        return DualHeadDataset(
            df['text_clean'].values, df[label_col].values,
            df['is_hateful'].values, df['is_sarcastic'].values,
            tokenizer, MAX_LEN)

    train_loader = DataLoader(make_ds(train_df), BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(make_ds(val_df),   BATCH_SIZE*2)
    test_loader  = DataLoader(make_ds(test_df),  64)

    model = AblationModel(MODEL_NAME,
                           use_hate_head=use_hate,
                           use_sarc_head=use_sarc)
                           
    if torch.cuda.device_count() > 1:
        print(f"Let's use {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
        
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, len(train_loader)//10, len(train_loader)*EPOCHS)

    crit_3 = nn.CrossEntropyLoss()
    crit_b = nn.BCEWithLogitsLoss()

    best_f1, best_state = 0, None

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            l3   = batch['labels_3class'].to(device)
            lh   = batch['labels_hate'].to(device)
            ls   = batch['labels_sarc'].to(device)

            optimizer.zero_grad()
            outputs = model(ids, mask)
            
            loss = GAMMA * crit_3(outputs['logits_3class'], l3)
            if 'hate_logit' in outputs: 
                loss += ALPHA * crit_b(outputs['hate_logit'], lh)
            if 'sarc_logit' in outputs: 
                loss += BETA  * crit_b(outputs['sarc_logit'], ls)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if scheduler: scheduler.step()
            total_loss += loss.item()

        model.eval()
        preds_v, labels_v = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                
                outputs = model(ids, mask)
                preds_v.extend(outputs['logits_3class'].argmax(1).cpu().numpy())
                
                labels_v.extend(batch['labels_3class'].numpy())

        f1 = f1_score(labels_v, preds_v, average='macro')
        if f1 > best_f1:
            best_f1   = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f} — Val F1: {f1:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    preds_t, labels_t = [], []
    with torch.no_grad():
        for batch in test_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            outputs = model(ids, mask)
            preds_t.extend(outputs['logits_3class'].argmax(1).cpu().numpy())
            
            labels_t.extend(batch['labels_3class'].numpy())

    acc = accuracy_score(labels_t, preds_t)
    f1  = f1_score(labels_t, preds_t, average='macro')
    print(f"\n[TEST] {variant_name} — {lang}")
    print(f"Accuracy: {acc:.4f}  Macro F1: {f1:.4f}")
    
    print(classification_report(labels_t, preds_t,
                                target_names=list(CLASS_NAMES.values())))

    import json
    
    variant_dir = f"{OUTPUT_DIR}{variant_name}_{lang}/"
    os.makedirs(variant_dir, exist_ok=True)
    
    torch.save(best_state, f"{variant_dir}model_weights.pth")
    np.save(f"{variant_dir}preds.npy", np.array(preds_t))
    np.save(f"{variant_dir}labels.npy", np.array(labels_t))
    
    config_info = {
        "model_name": MODEL_NAME,
        "variant": variant_name,
        "lang": lang,
        "use_hate_head": use_hate,
        "use_sarc_head": use_sarc,
        "test_accuracy": acc,
        "test_macro_f1": f1
    }
    with open(f"{variant_dir}config.json", "w") as f:
        json.dump(config_info, f, indent=4)
        
    del model; gc.collect(); torch.cuda.empty_cache()
    return {'variant': variant_name, 'lang': lang,
            'test_acc': round(acc,4), 'test_f1': round(f1,4)}

In [6]:
# ============================================================
# Run ablations for Bangla
# ============================================================
ablation_results = []

# Variant 1: No hate head
r = run_ablation(bn_train, bn_val, bn_test, 'label', 'Bangla',
                 use_hate=False, use_sarc=True,
                 variant_name='No_HateHead')
ablation_results.append(r)

# Variant 2: No sarcasm head
r = run_ablation(bn_train, bn_val, bn_test, 'label', 'Bangla',
                 use_hate=True, use_sarc=False,
                 variant_name='No_SarcHead')
ablation_results.append(r)

# Variant 3: No auxiliary heads (single-task)
r = run_ablation(bn_train, bn_val, bn_test, 'label', 'Bangla',
                 use_hate=False, use_sarc=False,
                 variant_name='SingleTask_XLM-R')
ablation_results.append(r)

results_df = pd.DataFrame(ablation_results)
results_df.to_csv(f"{OUTPUT_DIR}final_ablation_results_bn.csv", index=False)


Ablation: No_HateHead — Bangla
  use_hate_head=False  use_sarc_head=True


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Let's use 2 GPUs!
  Epoch 1/5 — Loss: 0.3670 — Val F1: 0.7583
  Epoch 2/5 — Loss: 0.2430 — Val F1: 0.7860
  Epoch 3/5 — Loss: 0.2083 — Val F1: 0.7864
  Epoch 4/5 — Loss: 0.1848 — Val F1: 0.7980
  Epoch 5/5 — Loss: 0.1671 — Val F1: 0.8034

[TEST] No_SarcHead — Bangla
Accuracy: 0.8255  Macro F1: 0.8055
              precision    recall  f1-score   support

 Non-hateful       0.83      0.84      0.83      4140
     Hateful       0.87      0.92      0.90      2560
   Sarcastic       0.72      0.65      0.69      1677

    accuracy                           0.83      8377
   macro avg       0.81      0.80      0.81      8377
weighted avg       0.82      0.83      0.82      8377


Ablation: SingleTask_XLM-R — Bangla
  use_hate_head=False  use_sarc_head=False


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Let's use 2 GPUs!
  Epoch 1/5 — Loss: 0.2822 — Val F1: 0.7426
  Epoch 2/5 — Loss: 0.1898 — Val F1: 0.7918
  Epoch 3/5 — Loss: 0.1636 — Val F1: 0.8059
  Epoch 4/5 — Loss: 0.1466 — Val F1: 0.8010
  Epoch 5/5 — Loss: 0.1329 — Val F1: 0.8083

[TEST] SingleTask_XLM-R — Bangla
Accuracy: 0.8286  Macro F1: 0.8092
              precision    recall  f1-score   support

 Non-hateful       0.83      0.84      0.84      4140
     Hateful       0.88      0.92      0.90      2560
   Sarcastic       0.73      0.66      0.69      1677

    accuracy                           0.83      8377
   macro avg       0.81      0.81      0.81      8377
weighted avg       0.83      0.83      0.83      8377

